# Vernier Radiation Monitor with a LabQuest

This is an example of how to use a Vernier Radiation Monitor with a LabQuest data acquisition device over USB.   

## Installing the LabQuest Package
A couple of python packages are required to use this code.  Most packages should already be installed with the standard, Anaconda installation.   The one exception is the "LabQuest" package.   To add this package to your python installation, use these steps:

1. Open the **Anaconda Navigator** application (Windows) or **Terminal** (Mac OS).
2. Type: 'pip install labquest'
   


In [ ]:
from labquest import LabQuest
import time
import sys
from tqdm import tqdm # for progress bar

# Helper Functions

These are some examples of helper functions that simplify the task of reading data from the USB hardware. 

In [ ]:
def InitRadMonitor(LabQuest, channel='dig1', debug=False):
    """
    Function to initialize the radiation monitor.

    Digital sensors tend to have some garbage stuck in their buffer
    when they are first initialized.   This initialization tries to
    gracefully handle the first several 'garbage' reads from the radiation monitor.

    Parameters:
        LabQuest (LabQuest class)
        channel (string) - the name of the LabQuest channel that the
            sensor is plugged into.  We assume that the sensor is plugged
            into dig1. 

    Returns:
        integer - the number of counts cleared from the sensor during
            initialization.   This number should be subtracted from the
            'real' measurement.

    Note that the Radiation Monitor does NOT clear the number of counts from 
    memory after each successive read().   The sensor will continue to 
    increment the number of observed counts until the stop() command is issued. 
    
    """
    MAXTRIES=50  # number of reads before giving up
    tries=iter(range(MAXTRIES)) 
    digital_count = 0
    
    if (debug): print("Initializing the radiation monitor on %s.   Please wait..." % channel)
    for i in tries:
        digital_count = LabQuest.read(channel)
        if digital_count == None:
            print("   Error, no sensor to initialize!")
            sys.exit()
        elif digital_count != 0:
            if (debug): print("   SUCCESS!")
            return digital_count
            
    print("   ERROR: Failed to initialize!")
    sys.exit()

In [ ]:
def ReadRadMonitor(LabQuest, time, channel='dig1'):
    """
    Function to read the counts from a Vernier Radiation Monitor

    Parameters:
        LabQuest (LabQuest class)
        time (int) - integration time (in seconds) for the observation.
        channel (string) - channel that the radiation monitor is plugged into.

    Returns:
        integer - the number of observed counts during the specified time.
    """
    # first, initialize the sensor
    init_counts=InitRadMonitor(lq, channel)
    
    # now loop over successive sensor reads
    # tqdm provides a status bar to show the integration
    myformat='{l_bar}{bar}| {elapsed} / {total}'
    for i in tqdm(range(time), desc="Waiting", ascii=False, ncols=75, bar_format=myformat):
        dig1_measurement = lq.read(channel)

    # subtract the initialization before reporting the counts
    observed_counts = dig1_measurement - init_counts
    return observed_counts
    

In [ ]:
INTEGRATION_TIME=30 # integer seconds - this is the total observation time

READ_PERIOD=1000 # ms for each sensor read.  Leave this set to 1000ms (1s)

# create a LabQuest device to talk to
lq = LabQuest() 

# configure the LabQuest Settings
lq.open() 
lq.select_sensors(dig1='photogate_count')    # Configure dig1 as a photogate counter 
lq.start(READ_PERIOD)    # read time in ms.  THIS SHOULD BE 1000 for best performance 

# begin the observation
print("Begin initialization and %d second measurement" % (INTEGRATION_TIME))
observed_counts = ReadRadMonitor(lq,INTEGRATION_TIME)

# print the results
print()
print("Measurement Results:")
print("| Time (s) |\t Counts |\t Rate (cps) |")
print("=============================================")
print("| %8d |\t %6d |\t %10.4f |" % (INTEGRATION_TIME,observed_counts,observed_counts/INTEGRATION_TIME))

lq.stop() 
lq.close()